<a href="https://colab.research.google.com/github/mohadeseh-mansoorian/-9/blob/main/Another_copy_of_JKS_Certificate_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq install -y default-jdk
!pip -q install requests pandas openpyxl cryptography

import os
import re
import time
import tempfile
import subprocess
import requests
import pandas as pd

from urllib.parse import quote
from cryptography import x509
from cryptography.hazmat.primitives import hashes
from IPython.display import display

print("Java:", subprocess.run(
    ["java", "-version"], capture_output=True, text=True
).stderr.splitlines()[0])
print("آماده شروع هستیم.")

Java: openjdk version "21.0.12.1" 2026-08-18
آماده شروع هستیم.


In [ ]:

# سلول ۲: جست‌وجوی گسترده‌تر مخازن و فایل‌های JKS

API = "https://api.github.com"
session = requests.Session()
session.headers.update({"User-Agent": "JKS-Academic-Research"})

queries = [
    "jks java",
    "java keystore certificate",
    "jks test",
    "purejks",
    "keystore samples"
]

repositories = {}
candidates = []
scan_errors = []

# جست‌وجوی مخازن مرتبط
for query in queries:
    try:
        response = session.get(
            f"{API}/search/repositories",
            params={"q": query, "per_page": 30},
            timeout=25
        )

        if response.status_code != 200:
            print(f"Search HTTP {response.status_code}: {query}")
            continue

        for repo in response.json().get("items", []):
            repositories[repo["full_name"]] = {
                "branch": repo["default_branch"],
                "url": repo["html_url"]
            }

    except Exception as e:
        print("Search error:", str(e)[:150])

print("مخازن یکتا:", len(repositories))

# بررسی تمام مسیرهای هر مخزن؛ بدون فیلتر پوشه
for repo, info in repositories.items():
    branch = info["branch"]

    try:
        url = (
            f"{API}/repos/{repo}/git/trees/"
            f"{quote(branch, safe='')}?recursive=1"
        )
        response = session.get(url, timeout=30)

        if response.status_code != 200:
            scan_errors.append({
                "Repository": repo,
                "Error": f"HTTP {response.status_code}"
            })
            continue

        data = response.json()

        for item in data.get("tree", []):
            path = item.get("path", "")

            if (
                item.get("type") == "blob"
                and path.lower().endswith(".jks")
                and item.get("size", 0) <= 2_000_000
            ):
                candidates.append({
                    "Repository": repo,
                    "Branch": branch,
                    "Path": path,
                    "Size (bytes)": item.get("size", 0),
                    "Repository URL": info["url"],
                    "File URL": (
                        f"https://github.com/{repo}/blob/"
                        f"{quote(branch, safe='')}/"
                        f"{quote(path, safe='/')}"
                    )
                })

        if data.get("truncated"):
            scan_errors.append({
                "Repository": repo,
                "Error": "Tree truncated by GitHub API"
            })

        time.sleep(0.2)

    except Exception as e:
        scan_errors.append({
            "Repository": repo,
            "Error": str(e)[:200]
        })

candidates_df = pd.DataFrame(candidates)

if not candidates_df.empty:
    candidates_df = candidates_df.drop_duplicates(
        subset=["Repository", "Path"]
    ).reset_index(drop=True)

print("فایل‌های JKS پیدا‌شده:", len(candidates_df))
display(candidates_df)

if scan_errors:
    print("مخازن یا درخواست‌های ناموفق:")
    display(pd.DataFrame(scan_errors))

مخازن یکتا: 108
فایل‌های JKS پیدا‌شده: 88


,Repository,Branch,Path,Size (bytes),Repository URL,File URL
0,lenchv/jks-js,master,examples/assets/keystore.jks,2232,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/e...
1,lenchv/jks-js,master,examples/assets/truststore.jks,942,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/e...
2,lenchv/jks-js,master,tests/feature/java11/keystore/DSA_1024_keystor...,1623,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...
3,lenchv/jks-js,master,tests/feature/java11/keystore/EC_256_keystore.jks,1020,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...
4,lenchv/jks-js,master,tests/feature/java11/keystore/RSA_2048_MD5with...,2575,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...
...,...,...,...,...,...,...
83,prayagupa/tls.kotlin,master,client/conf/restapi.jks,873,https://github.com/prayagupa/tls.kotlin,https://github.com/prayagupa/tls.kotlin/blob/m...
84,prayagupa/tls.kotlin,master,server/conf/eccountKeyStore.jks,2215,https://github.com/prayagupa/tls.kotlin,https://github.com/prayagupa/tls.kotlin/blob/m...
85,prayagupa/tls.kotlin,master,server/conf/restapi.jks,1280,https://github.com/prayagupa/tls.kotlin,https://github.com/prayagupa/tls.kotlin/blob/m...
86,sanethmaduranga/test_data,main,client-truststore.jks,98778,https://github.com/sanethmaduranga/test_data,https://github.com/sanethmaduranga/test_data/b...


مخازن یا درخواست‌های ناموفق:


,Repository,Error
0,sabbirhsihah-web/app-testkey_untrusted.jks,HTTP 403
1,XuLianBao/jks,HTTP 403
2,iosToTop/JKsTest,HTTP 403
3,luisboch/cert-test,HTTP 403
4,Ezreal-y/jkstest,HTTP 403
5,GChan-Ch/JKSlibrary,HTTP 403
6,honor-hermit/TEST-PERU-EDU_VENTURE,HTTP 403
7,mOwlz/jkside,HTTP 403
8,jignesh-dadhaniya/jksolutions,HTTP 403
9,easingboy/JKSay,HTTP 403


In [ ]:

!pip -q install pyjks cryptography openpyxl

import warnings
import pandas as pd
import jks
import requests

from urllib.parse import quote
from cryptography import x509
from cryptography.hazmat.primitives import hashes

certificate_rows = []
scan_rows = []

def get_field(row, field, default=""):
    """دریافت امن مقدار یک ستون از جدول."""
    try:
        value = row[field]
        return default if pd.isna(value) else value
    except (KeyError, TypeError):
        return default

def format_name(name):
    """تبدیل نام گواهی به متن؛ بدون نمایش هشدارهای قالب‌بندی."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        try:
            return name.rfc4514_string()
        except Exception:
            return str(name)

def get_cert_dates(cert):
    """سازگار با نسخه‌های مختلف cryptography."""
    not_before = getattr(cert, "not_valid_before_utc", None)
    not_after = getattr(cert, "not_valid_after_utc", None)

    if not_before is None:
        not_before = cert.not_valid_before.replace(
            tzinfo=__import__("datetime").timezone.utc
        )

    if not_after is None:
        not_after = cert.not_valid_after.replace(
            tzinfo=__import__("datetime").timezone.utc
        )

    return not_before, not_after

for index, item in candidates_df.iterrows():
    repo = str(get_field(item, "Repository"))
    branch = str(get_field(item, "Branch", "main"))
    path = str(get_field(item, "Path"))
    repo_url = str(get_field(item, "Repository URL"))
    file_url = str(get_field(item, "File URL"))

    if not repo or not path:
        scan_rows.append({
            "Repository": repo,
            "Path": path,
            "Status": "Skipped: missing repository or path",
            "Certificate Count": 0,
            "Integrity Check": "Not performed",
            "Error": "Repository or file path is missing"
        })
        continue

    raw_url = (
        f"https://raw.githubusercontent.com/{repo}/"
        f"{quote(branch, safe='/')}/"
        f"{quote(path, safe='/')}"
    )

    if not file_url:
        file_url = raw_url

    found_count = 0

    try:
        response = session.get(raw_url, timeout=30)
        response.raise_for_status()
        data = response.content

        # رمز مخزن ارائه نمی‌شود؛ صحت MAC مخزن بررسی نخواهد شد.
        # کلیدهای خصوصی رمزگشایی نمی‌شوند.
        ks = jks.KeyStore.loads(
            data,
            store_password=None,
            try_decrypt_keys=False
        )

        entries = []

        # گواهی‌های مستقل
        for alias, entry in ks.certs.items():
            entries.append((
                alias,
                entry.cert,
                "TrustedCertEntry"
            ))

        # زنجیره گواهی‌های همراه کلید خصوصی
        for alias, entry in ks.private_keys.items():
            for chain_index, (cert_type, cert_data) in enumerate(
                entry.cert_chain or []
            ):
                entries.append((
                    f"{alias}_chain_{chain_index}",
                    cert_data,
                    "PrivateKeyEntry certificate"
                ))

        # بررسی گواهی‌ها، بدون استخراج کلید خصوصی
        for alias, cert_data, entry_type in entries:
            try:
                cert = x509.load_der_x509_certificate(cert_data)
                not_before, not_after = get_cert_dates(cert)

                now = pd.Timestamp.now(tz="UTC")
                if now < pd.Timestamp(not_before):
                    validity = "Not yet valid"
                elif now > pd.Timestamp(not_after):
                    validity = "Expired"
                else:
                    validity = "Within date range"

                fingerprint = cert.fingerprint(
                    hashes.SHA256()
                ).hex(":").upper()

                certificate_rows.append({
                    "Repository": repo,
                    "Repository URL": repo_url,
                    "File Path": path,
                    "File URL": file_url,
                    "Alias": alias,
                    "Entry Type": entry_type,
                    "Subject": format_name(cert.subject),
                    "Issuer": format_name(cert.issuer),
                    "Serial Number": format(cert.serial_number, "X"),
                    "Valid From UTC": not_before.isoformat(),
                    "Valid Until UTC": not_after.isoformat(),
                    "Validity Status": validity,
                    "SHA256 Fingerprint": fingerprint,
                    "Public Key Type": type(
                        cert.public_key()
                    ).__name__,
                    "Signature Algorithm": cert.signature_algorithm_oid.dotted_string,
                    "Integrity Check": "Skipped: no store password"
                })

                found_count += 1

            except Exception as cert_error:
                # خطای یک گواهی نباید کل مخزن را متوقف کند.
                certificate_rows.append({
                    "Repository": repo,
                    "Repository URL": repo_url,
                    "File Path": path,
                    "File URL": file_url,
                    "Alias": alias,
                    "Entry Type": entry_type,
                    "Status": "Certificate parsing failed",
                    "Error": str(cert_error)[:500],
                    "Integrity Check": "Skipped: no store password"
                })

        if found_count:
            status = "Parsed; integrity not verified"
            error_text = ""
        elif not entries:
            status = "Readable; no certificates found"
            error_text = ""
        else:
            status = "Parsed; no certificates could be processed"
            error_text = "Certificate parsing failed for all entries"

        scan_rows.append({
            "Repository": repo,
            "Repository URL": repo_url,
            "Path": path,
            "File URL": file_url,
            "File Size (bytes)": len(data),
            "Entry Count": len(ks.entries),
            "Certificate Count": found_count,
            "Status": status,
            "Integrity Check": "Skipped: no store password",
            "Error": error_text
        })

    except requests.RequestException as exc:
        scan_rows.append({
            "Repository": repo,
            "Repository URL": repo_url,
            "Path": path,
            "File URL": file_url,
            "Status": "Download failed",
            "Certificate Count": 0,
            "Integrity Check": "Not performed",
            "Error": str(exc)[:500]
        })

    except Exception as exc:
        scan_rows.append({
            "Repository": repo,
            "Repository URL": repo_url,
            "Path": path,
            "File URL": file_url,
            "Status": "Unreadable or unsupported JKS",
            "Certificate Count": 0,
            "Integrity Check": "Not verified",
            "Error": f"{type(exc).__name__}: {str(exc)[:500]}"
        })

    if (index + 1) % 10 == 0:
        print(
            f"بررسی {index + 1} از {len(candidates_df)} فایل انجام شد؛ "
            f"تعداد گواهی‌های استخراج‌شده: {len(certificate_rows)}"
        )

certificates_df = pd.DataFrame(certificate_rows)
scan_results_df = pd.DataFrame(scan_rows)

print("\n--- نتیجه نهایی ---")
print("فایل‌های بررسی‌شده:", len(scan_results_df))
print("گواهی‌های استخراج‌شده:", len(certificates_df))

if not scan_results_df.empty and "Status" in scan_results_df:
    print("\nوضعیت فایل‌ها:")
    print(scan_results_df["Status"].value_counts().to_string())

display(certificates_df.head())
display(scan_results_df.head())

/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": format_name(cert.subject),
/tmp/ipykernel_1085/1693683705.py:138: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Issuer": format_name(cert.issuer),
/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": format_name(cert.subject),
/tmp/ipykernel_1085/1693683705.py:138: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Issuer": format_name(cert.issuer),


بررسی 10 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 3


/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": format_name(cert.subject),
/tmp/ipykernel_1085/1693683705.py:138: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Issuer": format_name(cert.issuer),
/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": format_name(cert.subject),
/tmp/ipykernel_1085/1693683705.py:138: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Issuer": format_name(cert.issuer),
/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": format_name(cert.subject),
/tmp/ipykernel_1085/1693683705.py:138: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Issuer": format_name(cert.issuer),
/tmp/ipykernel_1085/1693683705.py:137: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  "Subject": form

بررسی 20 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 15


/tmp/ipykernel_1085/1693683705.py:115: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  cert = x509.load_der_x509_certificate(cert_data)
/tmp/ipykernel_1085/1693683705.py:139: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  "Serial Number": format(cert.serial_number, "X"),
/tmp/ipykernel_1085/1693683705.py:115: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  cert = x509.load_der_x509_certificate(cert_data)
/tmp/ipykernel_1085/1693683705.py:139: Cry

بررسی 30 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 167
بررسی 40 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 168
بررسی 50 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 168
بررسی 60 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 168
بررسی 70 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 168
بررسی 80 از 88 فایل انجام شد؛ تعداد گواهی‌های استخراج‌شده: 170

--- نتیجه نهایی ---
فایل‌های بررسی‌شده: 88
گواهی‌های استخراج‌شده: 261

وضعیت فایل‌ها:
Status
Unreadable or unsupported JKS                 57
Parsed; integrity not verified                26
Parsed; no certificates could be processed     3
Readable; no certificates found                1
Download failed                                1


/tmp/ipykernel_1085/1693683705.py:115: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  cert = x509.load_der_x509_certificate(cert_data)
/tmp/ipykernel_1085/1693683705.py:139: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  "Serial Number": format(cert.serial_number, "X"),
/tmp/ipykernel_1085/1693683705.py:115: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  cert = x509.load_der_x509_certificate(cert_data)
/tmp/ipykernel_1085/1693683705.py:139: Cry

,Repository,Repository URL,File Path,File URL,Alias,Entry Type,Subject,Issuer,Serial Number,Valid From UTC,Valid Until UTC,Validity Status,SHA256 Fingerprint,Public Key Type,Signature Algorithm,Integrity Check,Status,Error
0,lenchv/jks-js,https://github.com/lenchv/jks-js,examples/assets/keystore.jks,https://github.com/lenchv/jks-js/blob/master/e...,jks-js_chain_0,PrivateKeyEntry certificate,"CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...","CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...",4E4EAC98,2020-03-01T15:10:39+00:00,2020-05-30T15:10:39+00:00,Expired,38:1A:16:4F:D1:C8:B6:C5:82:A1:54:BC:2A:05:6E:C...,RSAPublicKey,1.2.840.113549.1.1.11,Skipped: no store password,NaN,NaN
1,lenchv/jks-js,https://github.com/lenchv/jks-js,examples/assets/truststore.jks,https://github.com/lenchv/jks-js/blob/master/e...,jks-js,TrustedCertEntry,"CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...","CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...",4E4EAC98,2020-03-01T15:10:39+00:00,2020-05-30T15:10:39+00:00,Expired,38:1A:16:4F:D1:C8:B6:C5:82:A1:54:BC:2A:05:6E:C...,RSAPublicKey,1.2.840.113549.1.1.11,Skipped: no store password,NaN,NaN
2,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/keystore/DSA_1024_keystore.jks,https://github.com/lenchv/jks-js/blob/master/t...,jks-js_chain_0,PrivateKeyEntry certificate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skipped: no store password,Certificate parsing failed,error parsing asn1 value: ParseError { kind: E...
3,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/keystore/EC_256_keystore.jks,https://github.com/lenchv/jks-js/blob/master/t...,jks-js_chain_0,PrivateKeyEntry certificate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skipped: no store password,Certificate parsing failed,error parsing asn1 value: ParseError { kind: E...
4,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/keystore/RSA_2048_MD5withRSA_key...,https://github.com/lenchv/jks-js/blob/master/t...,jks-js_chain_0,PrivateKeyEntry certificate,"CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...","CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-j...",4F159583,2021-03-14T14:04:10+00:00,2021-06-12T14:04:10+00:00,Expired,25:0C:0F:74:C1:FF:5B:F7:F7:86:91:B8:AF:01:8C:7...,RSAPublicKey,1.2.840.113549.1.1.4,Skipped: no store password,NaN,NaN


,Repository,Repository URL,Path,File URL,File Size (bytes),Entry Count,Certificate Count,Status,Integrity Check,Error
0,lenchv/jks-js,https://github.com/lenchv/jks-js,examples/assets/keystore.jks,https://github.com/lenchv/jks-js/blob/master/e...,2232.0,1.0,1,Parsed; integrity not verified,Skipped: no store password,
1,lenchv/jks-js,https://github.com/lenchv/jks-js,examples/assets/truststore.jks,https://github.com/lenchv/jks-js/blob/master/e...,942.0,1.0,1,Parsed; integrity not verified,Skipped: no store password,
2,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/java11/keystore/DSA_1024_keystor...,https://github.com/lenchv/jks-js/blob/master/t...,NaN,NaN,0,Unreadable or unsupported JKS,Not verified,BadKeystoreFormatException: Not a JKS or JCEKS...
3,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/java11/keystore/EC_256_keystore.jks,https://github.com/lenchv/jks-js/blob/master/t...,NaN,NaN,0,Unreadable or unsupported JKS,Not verified,BadKeystoreFormatException: Not a JKS or JCEKS...
4,lenchv/jks-js,https://github.com/lenchv/jks-js,tests/feature/java11/keystore/RSA_2048_MD5with...,https://github.com/lenchv/jks-js/blob/master/t...,NaN,NaN,0,Unreadable or unsupported JKS,Not verified,BadKeystoreFormatException: Not a JKS or JCEKS...


In [ ]:

output = "/content/jks_analysis.xlsx"

with pd.ExcelWriter(output, engine="openpyxl") as writer:
    certificates_df.to_excel(
        writer,
        sheet_name="Certificates",
        index=False
    )

    scan_results_df.to_excel(
        writer,
        sheet_name="Scan Results",
        index=False
    )

print("فایل اکسل آماده است:", output)
print("تعداد گواهی‌ها:", len(certificates_df))
print("تعداد فایل‌های بررسی‌شده:", len(scan_results_df))

from google.colab import files
files.download(output)

فایل اکسل آماده است: /content/jks_analysis.xlsx
تعداد گواهی‌ها: 261
تعداد فایل‌های بررسی‌شده: 88


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

!pip -q install pyjks cryptography

import jks
from cryptography import x509
from cryptography.hazmat.primitives import hashes

item = candidates_df.iloc[0]

raw_url = (
    f"https://raw.githubusercontent.com/{item['Repository']}/"
    f"{quote(item['Branch'], safe='/')}/"
    f"{quote(item['Path'], safe='/')}"
)

response = session.get(raw_url, timeout=30)
response.raise_for_status()
data = response.content

print("Repository:", item["Repository"])
print("File:", item["Path"])
print("Size:", len(data))

try:
    # بررسی ساختار بدون اعتبارسنجی رمز مخزن
    ks = jks.KeyStore.loads(
        data,
        store_password=None,
        try_decrypt_keys=False
    )

    print("نوع مخزن:", ks.store_type)
    print("تعداد کل ورودی‌ها:", len(ks.entries))
    print("تعداد گواهی‌های مستقل:", len(ks.certs))
    print("تعداد ورودی‌های کلید خصوصی:", len(ks.private_keys))

    extracted = []

    # گواهی‌های مستقل
    for alias, entry in ks.certs.items():
        extracted.append((alias, entry.cert, "TrustedCertEntry"))

    # گواهی‌های همراه کلید خصوصی؛ بدون رمزگشایی کلید
    for alias, entry in ks.private_keys.items():
        for index, (cert_type, cert_data) in enumerate(
            entry.cert_chain or []
        ):
            extracted.append((
                f"{alias}_chain_{index}",
                cert_data,
                "PrivateKeyEntry certificate"
            ))

    for alias, cert_data, entry_type in extracted:
        try:
            cert = x509.load_der_x509_certificate(cert_data)

            print("\nAlias:", alias)
            print("Entry type:", entry_type)
            print("Subject:", cert.subject.rfc4514_string())
            print("Issuer:", cert.issuer.rfc4514_string())
            print("Valid until:", cert.not_valid_after_utc.isoformat())
            print(
                "SHA256:",
                cert.fingerprint(hashes.SHA256()).hex(":").upper()
            )

        except Exception as cert_error:
            print("خطای خواندن گواهی:", alias, str(cert_error)[:200])

    print("\nتعداد گواهی‌های قابل پردازش:", len(extracted))

except Exception as e:
    print("تحلیل ساختاری ناموفق بود:", type(e).__name__, str(e)[:500])

Repository: lenchv/jks-js
File: examples/assets/keystore.jks
Size: 2232
نوع مخزن: jks
تعداد کل ورودی‌ها: 1
تعداد گواهی‌های مستقل: 0
تعداد ورودی‌های کلید خصوصی: 1

Alias: jks-js_chain_0
Entry type: PrivateKeyEntry certificate
Subject: CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-js,C=jks-js
Issuer: CN=jks-js,OU=jks-js,O=lenchv,L=jks-js,ST=jks-js,C=jks-js
Valid until: 2020-05-30T15:10:39+00:00
SHA256: 38:1A:16:4F:D1:C8:B6:C5:82:A1:54:BC:2A:05:6E:C9:69:78:BA:5F:46:71:91:0C:70:0A:CB:C7:85:3C:CD:62

تعداد گواهی‌های قابل پردازش: 1


/tmp/ipykernel_1085/1735651219.py:59: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  print("Subject:", cert.subject.rfc4514_string())
/tmp/ipykernel_1085/1735651219.py:60: UserWarning: Attribute's length must be >= 2 and <= 2, but it was 6
  print("Issuer:", cert.issuer.rfc4514_string())


In [ ]:

# تشخیص علت شکست تحلیل اولین فایل JKS

import os
import tempfile
import subprocess
from urllib.parse import quote

item = candidates_df.iloc[0]

raw_url = (
    f"https://raw.githubusercontent.com/{item['Repository']}/"
    f"{quote(item['Branch'], safe='/')}/"
    f"{quote(item['Path'], safe='/')}"
)

response = session.get(raw_url, timeout=30)
print("HTTP دانلود:", response.status_code)
response.raise_for_status()

with tempfile.TemporaryDirectory() as tmp:
    store = os.path.join(tmp, "sample.jks")

    with open(store, "wb") as f:
        f.write(response.content)

    print("اندازه فایل:", os.path.getsize(store), "bytes")
    print("هدر فایل:", response.content[:16].hex())

    for kind in ["JKS", "PKCS12"]:
        result = subprocess.run(
            [
                "keytool", "-list", "-v",
                "-keystore", store,
                "-storetype", kind,
                "-storepass", ""
            ],
            capture_output=True,
            text=True,
            timeout=20
        )

        print("\n--- نوع مخزن:", kind, "---")
        print("Exit code:", result.returncode)
        print("STDOUT:", result.stdout[:1200])
        print("STDERR:", result.stderr[:2000])

HTTP دانلود: 200
اندازه فایل: 2232 bytes
هدر فایل: feedfeed000000020000000100000001

--- نوع مخزن: JKS ---
Exit code: 1
STDOUT: [0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
keytool error: java.io.IOException: Keystore was tampered with, or password was incorrect
java.io.IOException: Keystore was tampered with, or password was incorrect
	at java.base/sun.security.provider.JavaKeyStore.engineLoad(JavaKeyStore.java:804)
	at java.base/sun.security.util.KeyStoreDelegator.engineLoad(KeyStoreDelegator.java:228)
	at java.base/java.security.KeyStore.load(KeyStore.java:1520)
	at java.base/sun.security.tools.keytool.Main.doCommands(Main.java:948)
	at java.base/sun.security.tools.keytool.Main.run(Main.java:419)
	at jav

In [ ]:
# ============================================================
# سلول جدید (آخرین سلول):
# دانلود تمام فایل‌های JKS بر اساس جدیدترین candidates_df
# و ساخت فایل ZIP
# ============================================================

import os
import io
import time
import zipfile
import requests
import pandas as pd
from urllib.parse import quote
from IPython.display import display


# --- تابع کمکی برای خواندن امن مقادیر ---
def _safe_str(value, default=""):
    try:
        if value is None or pd.isna(value):
            return default
        return str(value).strip()
    except Exception:
        return default


# --- ۱) بررسی وجود candidates_df ---
if 'candidates_df' not in globals() or not isinstance(candidates_df, pd.DataFrame) or candidates_df.empty:
    print("⚠️ candidates_df یافت نشد یا خالی است.")
    print("ابتدا سلول جست‌وجو و ساخت candidates_df را اجرا کنید، سپس دوباره این سلول را اجرا کنید.")
else:
    print(f"📦 تعداد فایل‌های JKS برای دانلود: {len(candidates_df)}")
    print(f"📅 زمان اجرا: {pd.Timestamp.now(tz='UTC').isoformat()}")
    print("─" * 60)

    # --- ۲) session مستقل برای دانلود (بدون دست‌زدن به session قبلی) ---
    dl_session = requests.Session()
    dl_session.headers.update({
        "User-Agent": "JKS-Academic-Research-Downloader"
    })

    # --- ۳) آماده‌سازی ZIP و لاگ ---
    zip_buffer = io.BytesIO()
    download_log = []
    success_count = 0
    fail_count = 0
    total_bytes = 0

    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for idx, row in candidates_df.iterrows():
            repo   = _safe_str(row.get("Repository"))
            branch = _safe_str(row.get("Branch"), "main") or "main"
            path   = _safe_str(row.get("Path"))
            file_url = _safe_str(row.get("File URL"))

            if not repo or not path:
                download_log.append({
                    "Repository": repo, "Branch": branch, "Path": path,
                    "File URL": file_url, "Status": "Skipped",
                    "Error": "Repository or path is missing", "Bytes": 0
                })
                fail_count += 1
                continue

            raw_url = (
                f"https://raw.githubusercontent.com/{repo}/"
                f"{quote(branch, safe='/')}/"
                f"{quote(path, safe='/')}"
            )

            try:
                resp = dl_session.get(raw_url, timeout=30)
                resp.raise_for_status()
                content = resp.content

                # ساخت مسیر یکتا داخل ZIP
                safe_repo   = repo.replace("/", "__")
                safe_branch = branch.replace("/", "__")
                safe_path   = path.replace(":", "_").replace("\\", "/")
                arcname = f"{safe_repo}/{safe_branch}/{safe_path}"

                zf.writestr(arcname, content)

                download_log.append({
                    "Repository": repo, "Branch": branch, "Path": path,
                    "File URL": file_url, "Status": "Downloaded",
                    "Error": "", "Bytes": len(content)
                })
                success_count += 1
                total_bytes += len(content)

            except requests.RequestException as exc:
                download_log.append({
                    "Repository": repo, "Branch": branch, "Path": path,
                    "File URL": file_url, "Status": "Download failed",
                    "Error": f"HTTP: {str(exc)[:200]}", "Bytes": 0
                })
                fail_count += 1
            except Exception as exc:
                download_log.append({
                    "Repository": repo, "Branch": branch, "Path": path,
                    "File URL": file_url, "Status": "Error",
                    "Error": f"{type(exc).name}: {str(exc)[:200]}", "Bytes": 0
                })
                fail_count += 1

            if (idx + 1) % 10 == 0 or (idx + 1) == len(candidates_df):
                print(f"  ⏳ {idx + 1}/{len(candidates_df)} | "
                      f"موفق: {success_count} | ناموفق: {fail_count}")
                time.sleep(0.15)   # احترام به rate limit

    # --- ۴) ذخیرهٔ فایل ZIP ---
    zip_buffer.seek(0)
    zip_path = "/content/jks_files.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getvalue())

    zip_size = os.path.getsize(zip_path)
    report_df = pd.DataFrame(download_log)

    # --- ۵) نمایش نمونه‌ای از محتویات ZIP ---
    with zipfile.ZipFile(zip_path, "r") as zf_check:
        names = zf_check.namelist()
    print(f"\n📂 نمونه از محتویات ZIP (نمایش ۱۰ مورد از {len(names)}):")
    for n in names[:10]:
        print(f"   - {n}")
    if len(names) > 10:
        print(f"   ... و {len(names) - 10} فایل دیگر")

    # --- ۶) گزارش نهایی ---
    print("\n" + "=" * 60)
    print("📊 گزارش نهایی دانلود")
    print("=" * 60)
    print(f"✅ دانلود موفق:    {success_count}")
    print(f"❌ دانلود ناموفق:  {fail_count}")
    print(f"📦 حجم فایل ZIP:  {zip_size:,} بایت ({zip_size/1024:.2f} KB)")
    print(f"📁 مسیر فایل:     {zip_path}")
    print("\n📈 توزیع وضعیت:")
    print(report_df["Status"].value_counts().to_string())

    if not report_df[report_df["Status"] != "Downloaded"].empty:
        print("\n⚠️ جزئیات فایل‌های ناموفق:")
        display(report_df[report_df["Status"] != "Downloaded"])

    # --- ۷) ذخیرهٔ گزارش CSV ---
    report_csv_path = "/content/jks_download_report.csv"
    report_df.to_csv(report_csv_path, index=False, encoding="utf-8-sig")
    print(f"\n💾 گزارش CSV ذخیره شد: {report_csv_path}")

    # --- ۸) دانلود از Colab ---
    try:
        from google.colab import files as colab_files
        print("\n⬇️ شروع دانلود ZIP از Colab...")
        colab_files.download(zip_path)
        # برای دانلود گزارش CSV هم، خط زیر را فعال کنید:
        # colab_files.download(report_csv_path)
    except ImportError:
        print("ℹ️ خارج از Colab اجرا می‌شود؛ فایل ZIP در مسیر محلی آماده است.")

📦 تعداد فایل‌های JKS برای دانلود: 88
📅 زمان اجرا: 2026-11-04T16:55:33.922249+00:00
────────────────────────────────────────────────────────────
  ⏳ 10/88 | موفق: 10 | ناموفق: 0
  ⏳ 20/88 | موفق: 20 | ناموفق: 0
  ⏳ 30/88 | موفق: 30 | ناموفق: 0
  ⏳ 40/88 | موفق: 40 | ناموفق: 0
  ⏳ 50/88 | موفق: 50 | ناموفق: 0
  ⏳ 60/88 | موفق: 60 | ناموفق: 0
  ⏳ 70/88 | موفق: 70 | ناموفق: 0
  ⏳ 80/88 | موفق: 80 | ناموفق: 0
  ⏳ 88/88 | موفق: 88 | ناموفق: 0

📂 نمونه از محتویات ZIP (نمایش ۱۰ مورد از 88):
   - lenchv__jks-js/master/examples/assets/keystore.jks
   - lenchv__jks-js/master/examples/assets/truststore.jks
   - lenchv__jks-js/master/tests/feature/java11/keystore/DSA_1024_keystore.jks
   - lenchv__jks-js/master/tests/feature/java11/keystore/EC_256_keystore.jks
   - lenchv__jks-js/master/tests/feature/java11/keystore/RSA_2048_MD5withRSA_keystore.jks
   - lenchv__jks-js/master/tests/feature/java11/keystore/RSA_2048_SHA1withRSA_keystore.jks
   - lenchv__jks-js/master/tests/feature/java11/keystore/RSA_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# مقایسهٔ دقیق‌تر ZIP با اکسل (با نادیده گرفتن نام شاخه)
# ============================================================

import zipfile
import pandas as pd

ZIP_PATH = "/content/jks_files.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zip_names = zf.namelist()
zip_set = set(zip_names)

print(f"📦 تعداد فایل داخل ZIP: {len(zip_names)}")

if "scan_results_df" not in globals():
    print("⚠️ scan_results_df در حافظه نیست.")
else:
    print(f"📊 تعداد ردیف اکسل: {len(scan_results_df)}")

    # استخراج نام مخزن و مسیر از فایل‌های داخل ZIP
    # ساختار ZIP: repo__name/branch/path/to/file.jks
    zip_pairs = set()
    for name in zip_names:
        parts = name.split('/', 2) # [repo_name, branch, path]
        if len(parts) == 3:
            repo_part = parts[0].replace('__', '/')
            path_part = parts[2]
            zip_pairs.add((repo_part, path_part))

    # بررسی ردیف‌های اکسل
    missing_in_zip = 0
    found_in_zip = 0

    for _, row in scan_results_df.iterrows():
        repo = str(row.get("Repository", "")).strip()
        path = str(row.get("Path", "")).strip()

        if (repo, path) in zip_pairs:
            found_in_zip += 1
        else:
            missing_in_zip += 1
            # اگر خواستید ببینید کدام‌ها جا افتاده‌اند، خط زیر را از کامنت خارج کنید:
            # print(f"❌ جا افتاده: {repo} :: {path}")

    print("\n" + "=" * 60)
    print("📋 گزارش مقایسه نهایی")
    print("=" * 60)
    print(f"  کل ردیف‌های اکسل:         {len(scan_results_df)}")
    print(f"  کل فایل‌های ZIP:          {len(zip_names)}")
    print(f"  تطابق کامل (اکسل و ZIP):  {found_in_zip}  ✅")
    print(f"  در اکسل هست، در ZIP نیست: {missing_in_zip}  ❌")

    if missing_in_zip == 0:
        print("\n✅ عالی! تمام فایل‌های اکسل دقیقاً در ZIP دانلود شده‌اند.")
    else:
        print(f"\n⚠️ {missing_in_zip} فایل در اکسل هست ولی دانلود نشده است.")
    print("=" * 60)

📦 تعداد فایل داخل ZIP: 88
📊 تعداد ردیف اکسل: 88

📋 گزارش مقایسه نهایی
  کل ردیف‌های اکسل:         88
  کل فایل‌های ZIP:          88
  تطابق کامل (اکسل و ZIP):  88  ✅
  در اکسل هست، در ZIP نیست: 0  ❌

✅ عالی! تمام فایل‌های اکسل دقیقاً در ZIP دانلود شده‌اند.


In [ ]:
# ============================================================
# بررسی دقیق فایل‌های دانلود نشده و نمایش دلیل خطا
# ============================================================

import zipfile
import pandas as pd

ZIP_PATH = "/content/jks_files.zip"
REPORT_PATH = "/content/jks_download_report.csv"

# ۱) خواندن نام فایل‌های داخل ZIP
try:
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zip_names = zf.namelist()
    zip_pairs = set()
    for name in zip_names:
        parts = name.split('/', 2)
        if len(parts) == 3:
            repo_part = parts[0].replace('__', '/')
            path_part = parts[2]
            zip_pairs.add((repo_part, path_part))
except Exception as e:
    print(f"خطا در خواندن فایل ZIP: {e}")
    zip_pairs = set()

# ۲) خواندن گزارش دانلود (که در کد اصلی دانلود ساخته شد)
try:
    report_df = pd.read_csv(REPORT_PATH)
except Exception as e:
    print(f"خطا در خواندن گزارش CSV: {e}")
    report_df = pd.DataFrame()

# ۳) بررسی شرطی
if "scan_results_df" not in globals():
    print("⚠️ متغیر scan_results_df در حافظه نیست.")
else:
    missing_files = []

    # پیدا کردن فایل‌هایی که در اکسل هستند ولی در ZIP نیستند
    for _, row in scan_results_df.iterrows():
        repo = str(row.get("Repository", "")).strip()
        path = str(row.get("Path", "")).strip()

        if (repo, path) not in zip_pairs:
            missing_files.append((repo, path))

    # چاپ نتیجه
    if len(missing_files) == 0:
        print("✅ هیچ فایلی جا نیفتاده است. تمام فایل‌های اکسل در ZIP موجود هستند.")
    else:
        print(f"⚠️ تعداد {len(missing_files)} فایل در اکسل هست ولی در ZIP نیست. دلیل دانلود نشدن آن‌ها:\n")
        print("-" * 60)

        for repo, path in missing_files:
            print(f"❌ فایل: {repo} :: {path}")

            # جستجو در گزارش CSV برای پیدا کردن دلیل خطا
            if not report_df.empty:
                match = report_df[(report_df["Repository"] == repo) & (report_df["Path"] == path)]
                if not match.empty:
                    status = match.iloc[0].get("Status", "نامشخص")
                    error_msg = match.iloc[0].get("Error", "بدون متن خطا")
                    print(f"   🔸 وضعیت: {status}")
                    print(f"   🔸 دلیل:  {error_msg}")
                else:
                    print("   🔸 دلیل: در فایل گزارش CSV پیدا نشد.")
            print("-" * 60)

✅ هیچ فایلی جا نیفتاده است. تمام فایل‌های اکسل در ZIP موجود هستند.
